In [18]:
import unicodedata
from pathlib import Path

import pandas as pd
import numpy as np
from scipy.stats import spearmanr

In [19]:
TARGET_WORDS = [
    "überspannen",
    "Manschette",
    "Fuß",
    "Rezeption",
    "abgebrüht",
    "Dynamik",
    "Engpaß",
    "abbauen",
    "Mißklang",
    "Abgesang",
    "Knotenpunkt",
    "Spielball",
    "zersetzen",
    "Armenhaus",
    "Ohrwurm",
    "Eintagsfliege",
    "Seminar",
    "Sensation",
    "Titel",
    "Schmiere",
    "ausspannen",
    "packen",
    "artikulieren",
    "abdecken",
]

In [20]:
annotations_schemas = {
    "en-en": "schemas_for_german/en-en/german",
    "ru-ru": "schemas_for_german/ru-ru/german",
    "rusemshift-finetune": "schemas_for_german/rusemshift/finetune/german",
    "rusemshift-train": "schemas_for_german/rusemshift/train/german",
}

In [ ]:
base = Path("schemas_for_german/en-en/german")
print("Actual folder names on disk:")
for subdir in sorted(base.iterdir()):
    if subdir.is_dir():
        print(f"  repr: {repr(subdir.name)}")

In [ ]:
def build_schema_index(schema_base_path: str) -> dict:
    base = Path(schema_base_path)
    index = {}

    for subdir in base.iterdir():
        if subdir.is_dir():
            normalized = unicodedata.normalize("NFC", subdir.name)
            index[normalized] = subdir

    return index

In [ ]:
def load_scores_for_word(
    schema_base_dir: str,
    word: str,
    index: dict,
) -> pd.DataFrame:
    
    normalized_word = unicodedata.normalize("NFC", word)
    word_dir = index.get(normalized_word)
    
    if word_dir is None:
        print(f"  Directory not found for: {word}")
        return None
        
    score_files = list(word_dir.glob("*.scores"))
    if not score_files:
        print(f"  No scores file in: {word_dir}")
        return None
    
    data = pd.read_json(score_files[0])
    data["score"] = data["score"].apply(lambda x: np.mean([float(v) for v in x]))
    return data

In [ ]:
# def load_scores_for_word(schema_base_path: str, word: str) -> pd.DataFrame:
#     score_file = Path(schema_base_path) / word / f"dev.{word}.scores"
#     if not score_file.exists():
#         word_nfc = unicodedata.normalize("NFC", word)
#         score_file = Path(schema_base_path) / word_nfc  /f"dev.{word_nfc}.scores"
#     if score_file.exists() is False:
#         print(f"  File not found: {word}")
#         return None

#     data = pd.read_json(score_file)
#     data["score"] = data["score"].apply(lambda x: np.mean([float(v) for v in x]))
#     return data

In [ ]:
def compute_apd(scores_df: pd.DataFrame) -> float:
    if scores_df is None or len(scores_df) == 0:
        return None

    return float((1 - scores_df["score"]).mean())

In [ ]:
results = {}

for schema_name, schema_path in annotations_schemas.items():
    print(f"\n=== {schema_name} ===")
    results[schema_name] = {}
    index = build_schema_index(schema_path)
    
    
    for word in TARGET_WORDS:
        scores_df = load_scores_for_word(schema_path, word, index)
        apd = compute_apd(scores_df)
        results[schema_name][word] = apd
        if apd is not None:
            print(f"  {word}: APD = {apd:.4f}")
        else:
            print(f"  {word}: no data")


=== en-en ===
  File not found: überspannen
  überspannen: no data
  Manschette: APD = 0.4903
  File not found: Fuß
  Fuß: no data
  Rezeption: APD = 0.5130
  File not found: abgebrüht
  abgebrüht: no data
  Dynamik: APD = 0.5089
  File not found: Engpaß
  Engpaß: no data
  abbauen: APD = 0.5086
  File not found: Mißklang
  Mißklang: no data
  Abgesang: APD = 0.4891
  Knotenpunkt: APD = 0.5032
  Spielball: APD = 0.4860
  zersetzen: APD = 0.5046
  Armenhaus: APD = 0.4953
  Ohrwurm: APD = 0.4944
  Eintagsfliege: APD = 0.4848
  Seminar: APD = 0.4961
  Sensation: APD = 0.4958
  Titel: APD = 0.5081
  Schmiere: APD = 0.5178
  ausspannen: APD = 0.5176
  packen: APD = 0.5184
  artikulieren: APD = 0.5064
  abdecken: APD = 0.5121

=== ru-ru ===
  File not found: überspannen
  überspannen: no data
  Manschette: APD = 0.4909
  File not found: Fuß
  Fuß: no data
  Rezeption: APD = 0.5033
  File not found: abgebrüht
  abgebrüht: no data
  Dynamik: APD = 0.4993
  File not found: Engpaß
  Engpaß: no 

In [ ]:
def load_gold_data(path: str) -> dict:
    df = pd.read_csv(path, sep="\t")
    try:
        return dict(zip(df["lemma"], df["change_graded"]))
    except KeyError:
        return dict(zip(df["word"], df["change_graded"]))


gold_data = load_gold_data("gold-data-de.csv")

print("Spearman correlations: ")
for schema_name in annotations_schemas:
    pairs = [
        (results[schema_name][w], gold_data[w])
        for w in TARGET_WORDS
        if results[schema_name].get(w) is not None and w in gold_data
    ]
    if len(pairs) < 2:
        print(f"  {schema_name}: not enough data")
        continue

    apd_values, gold_values = zip(*pairs)
    spearman, pvalue = spearmanr(gold_values, apd_values)
    print(f"  {schema_name}: Spearman = {spearman:.4f} (p = {pvalue:.4f})")

Spearman correlations: 
  en-en: Spearman = -0.0649 (p = 0.7918)
  ru-ru: Spearman = 0.0825 (p = 0.7372)
  rusemshift-finetune: Spearman = 0.1684 (p = 0.4907)
  rusemshift-train: Spearman = 0.0632 (p = 0.7973)


In [25]:
rows = []
for schema_name in annotations_schemas:
    for word in TARGET_WORDS:
        rows.append(
            {
                "schema": schema_name,
                "word": word,
                "apd": results[schema_name].get(word),
            }
        )

summary_df = pd.DataFrame(rows)
summary_df.pivot(index="word", columns="schema", values="apd").round(4)

schema,en-en,ru-ru,rusemshift-finetune,rusemshift-train
word,,,,
Abgesang,0.4891,0.4946,0.4972,0.4980
Armenhaus,0.4953,0.4935,0.5033,0.4944
Dynamik,0.5089,0.4993,0.5045,0.5035
Eintagsfliege,0.4848,0.4901,0.4937,0.4924
Engpaß,NaN,NaN,NaN,NaN
Fuß,NaN,NaN,NaN,NaN
Knotenpunkt,0.5032,0.4988,0.5075,0.5005
Manschette,0.4903,0.4909,0.4900,0.4950
Mißklang,NaN,NaN,NaN,NaN
